In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# CELL 1: use one GPU only. Run this FIRST in a fresh session.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
n = torch.cuda.device_count()
print("GPUs visible:", n)
print("GPU name:", torch.cuda.get_device_name(0))
if n != 1:
    raise RuntimeError("Stop the session, then run this cell first again.")
print("CELL 1 DONE")


GPUs visible: 1
GPU name: Tesla T4
CELL 1 DONE


In [3]:
# CELL 2: Install Unsloth + download SocraTeach
import sys, subprocess, os

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "--upgrade", "unsloth", "unsloth_zoo"
], check=True)

import torch
print("Torch:", torch.__version__)

from unsloth import FastLanguageModel
print("Unsloth installed successfully")

base = "https://raw.githubusercontent.com/Ljyustc/SocraticLM/main/data/"
out = "/kaggle/working/SocraTeach_multi.json"

subprocess.run(["wget", "-q", "-O", out, base + "SocraTeach_multi.json"], check=True)

mb = os.path.getsize(out) / 1e6
print("SocraTeach_multi.json MB:", round(mb, 1))

if mb < 50:
    raise RuntimeError("Download failed. Check Internet.")

print("CELL 2 DONE")

Torch: 2.11.0+cu130
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth installed successfully
SocraTeach_multi.json MB: 76.6
CELL 2 DONE


Update: multi-subject curriculum data
This notebook keeps the original MathDial/SocraTeach processing and Qwen + QLoRA setup. It adds 1,500 English-language CBSE/NCERT-aligned QA examples from FoundryAILabs/k12-indian-curriculum-4.9m using streaming, with subject metadata and a separate general-tutor system prompt. The deterministic question-level split is approximately 80% train, 10% validation, and 10% test. The original training downsampling has been disabled (TRAIN_FRACTION = 1.0).

The dataset card declares Apache-2.0. Review the dataset card and source attribution before redistribution. This is curriculum-aligned data, not a claim that every row is copied directly from an NCERT textbook.

In [4]:
# CELL 3: build train, val and test data (80/10/10, no question leakage)
import re, json, os, random, hashlib, collections
from datasets import load_dataset
from transformers import AutoTokenizer

SEED = 42
TRAIN_FRACTION = 1.0  # Use the complete 80% training pool; split_of() already creates 80/10/10.
MODEL_NAME = "Qwen/Qwen3.5-4B"
OUT_DIR = "/kaggle/working/data"
os.makedirs(OUT_DIR, exist_ok=True)

# Existing datasets
multi = json.load(open("/kaggle/working/SocraTeach_multi.json"))
md = load_dataset("eth-nlped/mathdial")

# Keep the original MathDial/SocraTeach tutor instruction unchanged.
SYSTEM = ("You are a patient Socratic math tutor. Guide the student with short questions and hints. "
          "Do not give the final answer before the student finds it. "
          "If the student makes a mistake, help them find it themselves.\n\nProblem: {q}")

# New instruction used only for the additional multi-subject curriculum examples.
NCERT_SYSTEM = (
    "You are a patient school tutor who can help with mathematics, science, social science, "
    "English, and other school subjects. Always respond in clear, simple English. "
    "Explain concepts accurately at a school student's level. For problem-solving questions, "
    "guide the student step by step with hints. For factual, conceptual, reading-comprehension, "
    "or language questions, answer directly and explain the reasoning. Do not assume every "
    "question is about mathematics.\n\nTopic or question: {q}"
)

def norm(t):
    return re.sub(r"\s+", " ", str(t).lower()).strip()

def qhash(q):
    return hashlib.md5(q.encode()).hexdigest()[:12]

def split_of(q):
    # Deterministic question-level split: 80% train, 10% validation, 10% test.
    h = int(hashlib.md5((str(SEED) + q).encode()).hexdigest(), 16) % 100
    return "test" if h < 10 else "val" if h < 20 else "train"

def build_socrateach():
    out, skipped = [], 0
    for pid, v in multi.items():
        q = v["question"].strip()
        for did, turns in v["dialogues"].items():
            msgs, ok = [], True
            for t in turns:
                s = str(t.get("system", "")).strip()
                if not s:
                    ok = False
                    break
                msgs.append(["assistant", s, True])
                if "user" in t:
                    u = str(t["user"]).strip()
                    if not u:
                        ok = False
                        break
                    msgs.append(["user", u, False])
            if ok and len(msgs) >= 2:
                out.append({"source": "socrateach", "id": did, "question": q, "turns": msgs})
            else:
                skipped += 1
    return out, skipped

TAG = re.compile(r"\((probing|generic|telling|focus)\)\s*(.*)$", re.S)

def parse_mathdial(conv):
    turns = []
    for t in conv.split("|EOM|"):
        t = t.strip()
        if not t:
            continue
        sp, _, rest = t.partition(":")
        rest = rest.strip()
        if not rest:
            continue
        if sp.strip() == "Teacher":
            m = TAG.match(rest)
            tag, text = (m.group(1), m.group(2).strip()) if m else ("NO_TAG", rest)
            if text:
                turns.append(["assistant", text, tag != "telling"])
        else:
            turns.append(["user", rest, False])
    merged = []
    for role, text, learn in turns:
        if merged and merged[-1][0] == role:
            merged[-1][1] += "\n" + text
            merged[-1][2] = merged[-1][2] and learn
        else:
            merged.append([role, text, learn])
    return merged

def build_mathdial():
    out, skipped = [], 0
    for s in ["train", "test"]:
        for r in md[s]:
            turns = parse_mathdial(r["conversation"])
            if any(t[0] == "assistant" and t[2] for t in turns):
                out.append({"source": "mathdial", "id": "md_" + str(len(out)),
                            "question": r["question"].strip(), "turns": turns})
            else:
                skipped += 1
    return out, skipped

st_all, st_skipped = build_socrateach()
md_all, md_skipped = build_mathdial()
print("SocraTeach kept:", len(st_all), "| skipped:", st_skipped)
print("MathDial kept:", len(md_all), "| skipped:", md_skipped)

pools = {"mathdial": collections.defaultdict(list), "socrateach": collections.defaultdict(list)}
for d in md_all:
    pools["mathdial"][split_of(norm(d["question"]))].append(d)
for d in st_all:
    pools["socrateach"][split_of(norm(d["question"]))].append(d)

rng = random.Random(SEED)
chosen = {sp: [] for sp in ["train", "val", "test"]}
for sp in ["train", "val", "test"]:
    n = min(len(pools["mathdial"][sp]), len(pools["socrateach"][sp]))
    if sp == "train":
        n = int(n * TRAIN_FRACTION)
    for src in ["mathdial", "socrateach"]:
        pool = sorted(pools[src][sp], key=lambda d: d["id"])
        chosen[sp] += rng.sample(pool, n)
    print("Dialogues chosen", sp, ":", n, "from each existing dataset")

# Add 1,500 English-language CBSE/NCERT-aligned examples.
# IMPORTANT: English is a folder in this repository, not a row-level language column.
# The actual JSONL schema uses instruction/output/subject/grade (not response/language).
NCERT_REPO = "FoundryAILabs/k12-indian-curriculum-4.9m"
NCERT_ENGLISH_TRAIN_URL = (
    "https://huggingface.co/datasets/FoundryAILabs/k12-indian-curriculum-4.9m/"
    "resolve/main/english/train.jsonl"
)
NCERT_TARGETS = {
    "science": 900,
    "social_science": 300,
    "mathematics": 150,
    "english": 75,   # 5% of the 1,500 added examples
    "other": 75
}
NCERT_TOTAL = sum(NCERT_TARGETS.values())

def subject_bucket(subject):
    s = norm(subject)
    if any(x in s for x in ["social science", "social studies", "history",
                            "geography", "civics", "political science", "economics"]):
        return "social_science"
    if any(x in s for x in ["english", "literature", "grammar", "language arts"]):
        return "english"
    if any(x in s for x in ["mathematics", "math", "arithmetic"]):
        return "mathematics"
    if any(x in s for x in ["science", "physics", "chemistry", "biology", "environment"]):
        return "science"
    return "other"

existing_questions = {
    norm(d["question"])
    for source_pool in pools.values()
    for split_pool in source_pool.values()
    for d in split_pool
}
ncert_seen = set()
ncert_selected = []
ncert_fallback = []
ncert_counts = collections.Counter()

print("Loading the English folder from the BharatLLM curriculum dataset...")
ncert_stream = load_dataset(
    "json",
    data_files={"train": NCERT_ENGLISH_TRAIN_URL},
    split="train",
    streaming=True
).shuffle(seed=SEED, buffer_size=20000)

MAX_SCAN = 500000
scan_i = -1
for scan_i, row in enumerate(ncert_stream):
    if scan_i >= MAX_SCAN or len(ncert_selected) >= NCERT_TOTAL:
        break

    # Dataset card's actual schema: instruction, output, subject, grade.
    # Keep response as a compatibility fallback, but do not filter on a language field:
    # the file itself is already english/train.jsonl.
    question = str(row.get("instruction") or "").strip()
    answer = str(row.get("output") or row.get("response") or "").strip()
    subject = str(row.get("subject") or "")
    if len(question) < 8 or len(answer) < 20:
        continue

    q_norm = norm(question)
    if q_norm in existing_questions or q_norm in ncert_seen:
        continue

    bucket = subject_bucket(subject)
    item = {
        "source": "ncert_" + bucket,
        "id": "ncert_" + str(len(ncert_selected) + len(ncert_fallback)),
        "question": question,
        "turns": [["user", question, False], ["assistant", answer, True]],
        "system_template": NCERT_SYSTEM
    }
    ncert_seen.add(q_norm)

    if ncert_counts[bucket] < NCERT_TARGETS[bucket]:
        ncert_selected.append(item)
        ncert_counts[bucket] += 1
    elif len(ncert_fallback) < NCERT_TOTAL:
        ncert_fallback.append(item)

if len(ncert_selected) < NCERT_TOTAL:
    needed = NCERT_TOTAL - len(ncert_selected)
    ncert_selected.extend(ncert_fallback[:needed])

if len(ncert_selected) < 1000:
    raise RuntimeError(
        f"Only {len(ncert_selected)} usable English curriculum examples were found. "
        f"Counts by subject: {dict(ncert_counts)}. "
        "Inspect a few rows from english/train.jsonl before training."
    )

ncert_selected = ncert_selected[:NCERT_TOTAL]
print("NCERT/CBSE-aligned examples selected:", len(ncert_selected))
print("NCERT subject counts:", dict(collections.Counter(x["source"] for x in ncert_selected)))
print("NCERT rows scanned:", scan_i + 1)

for d in ncert_selected:
    chosen[split_of(norm(d["question"]))].append(d)

def make_examples(d):
    system_template = d.get("system_template", SYSTEM)
    system = {"role": "system", "content": system_template.format(q=d["question"])}
    exs = []
    for i, (role, text, learn) in enumerate(d["turns"]):
        if role == "assistant" and learn:
            prompt = [system] + [{"role": r, "content": t} for r, t, _ in d["turns"][:i]]
            exs.append({"source": d["source"], "dialogue_id": d["id"],
                        "qkey": qhash(norm(d["question"])),
                        "prompt": prompt,
                        "completion": [{"role": "assistant", "content": text}]})
    return exs

data = {}
for sp in ["train", "val", "test"]:
    exs = []
    for d in chosen[sp]:
        exs += make_examples(d)
    random.Random(SEED).shuffle(exs)
    data[sp] = exs
    with open(OUT_DIR + "/" + sp + ".jsonl", "w") as f:
        for e in exs:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")

for sp in ["train", "val", "test"]:
    c = collections.Counter(e["source"] for e in data[sp])
    print(sp, "total:", len(data[sp]), "| sources:", dict(c))

total_examples = sum(len(data[sp]) for sp in data)
for sp in ["train", "val", "test"]:
    pct = 100 * len(data[sp]) / max(total_examples, 1)
    print(sp, "share:", round(pct, 2), "%")

keys = {sp: {e["qkey"] for e in data[sp]} for sp in data}
print("Leak check:", len(keys["train"] & keys["val"]),
      len(keys["train"] & keys["test"]), len(keys["val"] & keys["test"]))
print("Expected split: approximately 80% train, 10% validation, 10% test; leak counts should be 0 0 0.")
# Package the exact preprocessed data used by the trainer for project evidence.
import zipfile
manifest = {
    "seed": SEED,
    "split_method": "deterministic question-level hash split (approximately 80/10/10)",
    "train_fraction": TRAIN_FRACTION,
    "ncert_dataset": NCERT_REPO,
    "ncert_file": "english/train.jsonl",
    "ncert_selected": len(ncert_selected),
    "ncert_subject_counts": dict(collections.Counter(x["source"] for x in ncert_selected)),
    "split_example_counts": {sp: len(data[sp]) for sp in data},
    "source_example_counts": {sp: dict(collections.Counter(e["source"] for e in data[sp])) for sp in data},
    "question_hash_leak_counts": {
        "train_val": len(keys["train"] & keys["val"]),
        "train_test": len(keys["train"] & keys["test"]),
        "val_test": len(keys["val"] & keys["test"])
    }
}
with open(OUT_DIR + "/manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
with open(OUT_DIR + "/DATA_SOURCES_AND_LICENSES.txt", "w", encoding="utf-8") as f:
    f.write("MathDial: CC BY 4.0; cite the original MathDial paper and dataset.\n")
    f.write("SocraTeach_Multi: CC BY-NC 4.0; non-commercial restrictions apply.\n")
    f.write("BharatLLM K-12 curriculum dataset: dataset card declares Apache-2.0; third-party CBSE/NCERT-aligned data, not an official NCERT dump.\n")
    f.write("Check all upstream terms and provenance before redistribution or commercial use.\n")
zip_path = "/kaggle/working/preprocessed_tutor_data.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for name in ["train.jsonl", "val.jsonl", "test.jsonl", "manifest.json", "DATA_SOURCES_AND_LICENSES.txt"]:
        zf.write(OUT_DIR + "/" + name, arcname=name)
print("Preprocessed data package:", zip_path, "| MB:", round(os.path.getsize(zip_path)/1e6, 2))

print("CELL 3 DONE")


README.md:   0%|          | 0.00/1.93k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/7.18M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/1.86M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2262 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/599 [00:00<?, ? examples/s]

SocraTeach kept: 35149 | skipped: 2
MathDial kept: 2833 | skipped: 28
Dialogues chosen train : 2209 from each existing dataset
Dialogues chosen val : 324 from each existing dataset
Dialogues chosen test : 300 from each existing dataset
Loading the English folder from the BharatLLM curriculum dataset...
NCERT/CBSE-aligned examples selected: 1500
NCERT subject counts: {'ncert_social_science': 317, 'ncert_science': 900, 'ncert_mathematics': 208, 'ncert_english': 75}
NCERT rows scanned: 362366
train total: 24500 | sources: {'mathdial': 11700, 'socrateach': 11617, 'ncert_science': 708, 'ncert_social_science': 263, 'ncert_mathematics': 152, 'ncert_english': 60}
val total: 3580 | sources: {'mathdial': 1704, 'socrateach': 1720, 'ncert_science': 95, 'ncert_mathematics': 27, 'ncert_social_science': 27, 'ncert_english': 7}
test total: 3324 | sources: {'socrateach': 1569, 'mathdial': 1594, 'ncert_science': 97, 'ncert_english': 8, 'ncert_mathematics': 29, 'ncert_social_science': 27}
train share: 78

In [5]:

# CELL 4: Load Qwen3.5-4B using Unsloth + LoRA

import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer
from unsloth import FastLanguageModel

MODEL_NAME = "Qwen/Qwen3.5-4B"
DATA_DIR = "/kaggle/working/data"
MAX_TOKENS = 512

# Load prepared datasets
def load(sp):
    with open(f"{DATA_DIR}/{sp}.jsonl", "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

data = {
    sp: load(sp)
    for sp in ["train", "val", "test"]
}

# Load tokenizer
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

# Check that conversation contains a user message
def has_user_message(e):
    messages = e["prompt"] + e["completion"]
    return any(m.get("role") == "user" for m in messages)

# Count tokens using Qwen chat template
def n_tokens(e):
    messages = e["prompt"] + e["completion"]

    text = tok.apply_chat_template(
        messages,
        tokenize=False
    )

    return len(tok(text)["input_ids"])

# Filter invalid and over-length examples
train_ex = [
    e for e in data["train"]
    if has_user_message(e) and n_tokens(e) <= MAX_TOKENS
]

val_ex = [
    e for e in data["val"]
    if has_user_message(e) and n_tokens(e) <= MAX_TOKENS
]

test_ex = [
    e for e in data["test"]
    if has_user_message(e) and n_tokens(e) <= MAX_TOKENS
]

print("Train examples:", len(train_ex))
print("Validation examples:", len(val_ex))
print("Test examples:", len(test_ex))

# Convert to Hugging Face datasets
keep = ["prompt", "completion"]

train_ds = Dataset.from_list([
    {k: e[k] for k in keep} for e in train_ex
])

val_ds = Dataset.from_list([
    {k: e[k] for k in keep} for e in val_ex
])

test_ds = Dataset.from_list([
    {k: e[k] for k in keep} for e in test_ex
])

# Load Qwen3.5-4B through Unsloth
model, tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_TOKENS,
    dtype=torch.float16,
    load_in_4bit=False,
    full_finetuning=False,
)

# Configure LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.config.use_cache = False

print("GPU memory (GB):", round(
    torch.cuda.memory_allocated() / 1e9, 2
))

print("CELL 4 DONE")


config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

GPU memory (GB): 9.17
CELL 4 DONE


In [6]:

# CELL 5: Extended training configuration

import dataclasses
from trl import SFTConfig

wanted = dict(
    output_dir="/kaggle/working/checkpoints",

    max_steps=1500,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=75,

    fp16=False,
    bf16=False,
    gradient_checkpointing=False,

    max_length=MAX_TOKENS,
    completion_only_loss=True,

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,

    save_strategy="steps",
    save_steps=250,
    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
    seed=42,
)

fields = {f.name for f in dataclasses.fields(SFTConfig)}
missing = [k for k in wanted if k not in fields]

print("Unsupported arguments:", missing)

if missing:
    raise RuntimeError("Unsupported arguments found.")

args = SFTConfig(**wanted)

print("Training examples:", len(train_ds))
print("Validation examples:", len(val_ds))
print("Target optimizer steps:", 1500)
print("Effective batch size:", 16)
print("Approximate examples processed:", 24000)
print("CELL 5 DONE")


Unsupported arguments: []
Training examples: 15690
Steps per epoch: 981
Target optimizer steps: 180
Effective batch size: 16
CELL 5 DONE


In [7]:
# CELL 6: Train using Unsloth

import time, torch
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tok,
)

torch.cuda.reset_peak_memory_stats()
t0 = time.time()

trainer.train()

print("Training minutes:", round((time.time() - t0) / 60, 1))
print("Peak GPU GB:", round(
    torch.cuda.max_memory_allocated() / 1e9, 2
))

print("CELL 6 DONE")

`causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=2):   0%|          | 0/15690 [00:00<?, ? examples/s]

Unsloth: Warning: the tokenized prompt is not a prefix of the tokenized prompt + completion, usually because the prompt ends in a space or newline that merges with the completion's first token, or because the chat template renders the prompt alone differently. The completion is trained from where they diverge, as in TRL. To avoid it, end the prompt on a token boundary and start the completion with the whitespace instead ("Answer:" + " two").
Unsloth: Warning: the tokenized prompt is not a prefix of the tokenized prompt + completion, usually because the prompt ends in a space or newline that merges with the completion's first token, or because the chat template renders the prompt alone differently. The completion is trained from where they diverge, as in TRL. To avoid it, end the prompt on a token boundary and start the completion with the whitespace instead ("Answer:" + " two").


Unsloth: Building labels for train dataset (num_proc=2):   0%|          | 0/15690 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=2):   0%|          | 0/2276 [00:00<?, ? examples/s]

... (4 outputs left out)

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,1.591369
10,1.455227
15,1.138459
20,1.049675
25,1.126788
30,1.148816
35,0.973221
40,1.002903
45,1.012157
50,1.008076


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/checkpoints/checkpoint-30/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/checkpoints/checkpoint-60/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/checkpoints/checkpoint-90/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/checkpoints/checkpoint-120/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/checkpoints/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/checkpoints/checkpoint-180/tokenizer_config.json.


Training minutes: 53.7
Peak GPU GB: 9.61
CELL 6 DONE


In [8]:

# CELL 7: Save adapter + validation/test evaluation + perplexity

import os
import shutil
import math
import torch

ADAPTER_DIR = "/kaggle/working/lora_adapter"

# --------------------------------------------------
# 1. SAVE THE TRAINED LORA ADAPTER FIRST
# --------------------------------------------------

trainer.model.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)

print("Adapter saved:", ADAPTER_DIR)

# --------------------------------------------------
# 2. VALIDATION EVALUATION
# --------------------------------------------------

val_metrics = trainer.evaluate(
    eval_dataset=val_ds,
    metric_key_prefix="validation"
)

val_loss = val_metrics.get("validation_loss")

print("\n========== VALIDATION RESULTS ==========")

for k, v in val_metrics.items():
    if isinstance(v, (int, float)):
        print(k, "=", round(v, 4))

if val_loss is not None:
    print("Validation Perplexity =", round(math.exp(val_loss), 4))

# --------------------------------------------------
# 3. HELD-OUT TEST EVALUATION
# --------------------------------------------------

test_metrics = trainer.evaluate(
    eval_dataset=test_ds,
    metric_key_prefix="test"
)

test_loss = test_metrics.get("test_loss")

print("\n========== TEST RESULTS ==========")

for k, v in test_metrics.items():
    if isinstance(v, (int, float)):
        print(k, "=", round(v, 4))

if test_loss is not None:
    print("Test Perplexity =", round(math.exp(test_loss), 4))

# --------------------------------------------------
# 4. TRAINING LOSS HISTORY
# --------------------------------------------------

hist = trainer.state.log_history

train_losses = [
    (x["step"], round(x["loss"], 4))
    for x in hist if "loss" in x
]

print("\nTraining Loss History:")
print(train_losses)

# --------------------------------------------------
# 5. ZIP THE ADAPTER
# --------------------------------------------------

zip_path = "/kaggle/working/lora_adapter.zip"

shutil.make_archive(
    "/kaggle/working/lora_adapter",
    "zip",
    ADAPTER_DIR
)

mb = os.path.getsize(zip_path) / 1e6

print("\nAdapter ZIP:", zip_path)
print("ZIP Size:", round(mb, 2), "MB")

print("\nDownload lora_adapter.zip from Kaggle Output panel.")
print("CELL 7 DONE")


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/lora_adapter/tokenizer_config.json.


Adapter saved: /kaggle/working/lora_adapter


Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=2):   0%|          | 0/2276 [00:00<?, ? examples/s]

Unsloth: Warning: the tokenized prompt is not a prefix of the tokenized prompt + completion, usually because the prompt ends in a space or newline that merges with the completion's first token, or because the chat template renders the prompt alone differently. The completion is trained from where they diverge, as in TRL. To avoid it, end the prompt on a token boundary and start the completion with the whitespace instead ("Answer:" + " two").
Unsloth: Warning: the tokenized prompt is not a prefix of the tokenized prompt + completion, usually because the prompt ends in a space or newline that merges with the completion's first token, or because the chat template renders the prompt alone differently. The completion is trained from where they diverge, as in TRL. To avoid it, end the prompt on a token boundary and start the completion with the whitespace instead ("Answer:" + " two").


Unsloth: Building labels for eval dataset (num_proc=2):   0%|          | 0/2276 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2276 [00:00<?, ? examples/s]

... (5 outputs left out)

Unsloth: Building labels for eval dataset (num_proc=2):   0%|          | 0/2138 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2138 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step
1.027251,1.009552,180



========== TEST RESULTS ==========
test_loss = 1.0096
Test Perplexity = 2.7444

Training Loss History:
[(5, 1.5914), (10, 1.4552), (15, 1.1385), (20, 1.0497), (25, 1.1268), (30, 1.1488), (35, 0.9732), (40, 1.0029), (45, 1.0122), (50, 1.0081), (55, 0.9533), (60, 1.1142), (65, 0.9995), (70, 0.9234), (75, 0.9834), (80, 0.9336), (85, 0.9532), (90, 1.0423), (95, 0.9366), (100, 0.9758), (105, 1.051), (110, 0.8933), (115, 1.0002), (120, 1.0366), (125, 0.9464), (130, 0.9374), (135, 0.9798), (140, 0.911), (145, 0.9299), (150, 1.1002), (155, 0.9368), (160, 1.0494), (165, 0.9522), (170, 0.9097), (175, 0.8674), (180, 1.0273)]

Adapter ZIP: /kaggle/working/lora_adapter.zip
ZIP Size: 67.39 MB

Download lora_adapter.zip from Kaggle Output panel.
CELL 7 DONE


In [10]:

# CELL 8: Text-only qualitative checks

import torch

m = trainer.model
m.eval()
m.config.use_cache = True

check_sources = [
    "mathdial",
    "socrateach",
    "ncert_science",
    "ncert_social_science",
    "ncert_english",
    "ncert_mathematics"
]

def get_text_messages(messages):
    cleaned = []

    for msg in messages:
        content = msg.get("content", "")

        # Skip examples containing non-text content
        if not isinstance(content, str):
            return None

        cleaned.append({
            "role": msg["role"],
            "content": content
        })

    return cleaned

for src in check_sources:

    matches = [
        x for x in test_ex
        if x.get("source") == src
        and len(x["prompt"]) >= 2
    ]

    found = False

    for e in matches:
        prompt = get_text_messages(e["prompt"])
        completion = get_text_messages(e["completion"])

        if prompt is None or completion is None:
            continue

        try:
            text = tok.apply_chat_template(
                prompt,
                tokenize=False,
                add_generation_prompt=True
            )

            # Explicitly use the underlying text tokenizer
            base_tok = tok.tokenizer if hasattr(tok, "tokenizer") else tok

            inputs = base_tok(
                text,
                return_tensors="pt"
            ).to(m.device)

            with torch.no_grad():
                out = m.generate(
                    **inputs,
                    max_new_tokens=100,
                    do_sample=False,
                    pad_token_id=base_tok.pad_token_id,
                    eos_token_id=base_tok.eos_token_id
                )

            answer = base_tok.decode(
                out[0][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True
            )

            print("\n===== After training:", src, "=====")
            print("Student question:", prompt[-1]["content"][:300])
            print("Reference answer:", completion[0]["content"][:300])
            print("Model answer:", answer[:500])

            found = True
            break

        except Exception as ex:
            print("Skipping problematic example:", str(ex)[:200])

    if not found:
        print("No usable text-only example found for:", src)

print("\nCELL 8 DONE")


`causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.



===== After training: mathdial =====
Student question: The group produces an additional chair every 6 hours.
Reference answer: So how many additional chairs are produced in the first 6 hours?
Model answer: So how many additional chairs would they produce in 6 hours?

===== After training: socrateach =====
Student question: The total cost of the field trip is $140, because each student pays $7, and there are 20 students.
Reference answer: Well done! Now, if the field trip cost $140 and the class had $150 in total, how much is left in Miss Grayson's class fund after paying for the trip?
Model answer: That's correct! Now, if the class had $150 in total and the field trip cost $140, how much is left in Miss Grayson's class fund?

===== After training: ncert_science =====
Student question: Application Problem: Why are fibers like Nylon used to make ropes and carpets?
Reference answer: Fibers have strong intermolecular forces like hydrogen bonding. This leads to close packing of chains, hig

In [2]:
# !kaggle kernels output vimalpandey6388/socraticncrtkaggle \
#     -p /kaggle/working/recovered_output

Output file downloaded to /kaggle/working/recovered_output/SocraTeach_multi.json
Output file downloaded to /kaggle/working/recovered_output/checkpoints/README.md
Output file downloaded to /kaggle/working/recovered_output/checkpoints/checkpoint-1050/README.md
Output file downloaded to /kaggle/working/recovered_output/checkpoints/checkpoint-1050/adapter_config.json
Output file downloaded to /kaggle/working/recovered_output/checkpoints/checkpoint-1050/adapter_model.safetensors
Output file downloaded to /kaggle/working/recovered_output/checkpoints/checkpoint-1050/chat_template.jinja
Output file downloaded to /kaggle/working/recovered_output/checkpoints/checkpoint-1050/optimizer.pt
Output file downloaded to /kaggle/working/recovered_output/checkpoints/checkpoint-1050/rng_state.pth
Output file downloaded to /kaggle/working/recovered_output/checkpoints/checkpoint-1050/scaler.pt
Output file downloaded to /kaggle/working/recovered_output/checkpoints/checkpoint-1050/scheduler.pt
Output file down

In [3]:
# !pip uninstall -y torchao

In [4]:
# import os
# import shutil

# source = "/kaggle/working/recovered_output"

# print("Downloaded files:")
# for root, dirs, files in os.walk(source):
#     for file in files:
#         print(os.path.join(root, file))

# shutil.make_archive(
#     "/kaggle/working/recovered_checkpoints",
#     "zip",
#     source
# )

# print("ZIP CREATED")

Downloaded files:
/kaggle/working/recovered_output/preprocessed_tutor_data.zip
/kaggle/working/recovered_output/socraticncrtkaggle.log
/kaggle/working/recovered_output/SocraTeach_multi.json
/kaggle/working/recovered_output/checkpoints/README.md
/kaggle/working/recovered_output/checkpoints/checkpoint-1100/scheduler.pt
/kaggle/working/recovered_output/checkpoints/checkpoint-1100/optimizer.pt
/kaggle/working/recovered_output/checkpoints/checkpoint-1100/tokenizer_config.json
/kaggle/working/recovered_output/checkpoints/checkpoint-1100/rng_state.pth
/kaggle/working/recovered_output/checkpoints/checkpoint-1100/chat_template.jinja
/kaggle/working/recovered_output/checkpoints/checkpoint-1100/adapter_model.safetensors
/kaggle/working/recovered_output/checkpoints/checkpoint-1100/tokenizer.json
/kaggle/working/recovered_output/checkpoints/checkpoint-1100/adapter_config.json
/kaggle/working/recovered_output/checkpoints/checkpoint-1100/training_args.bin
/kaggle/working/recovered_output/checkpoints/

In [5]:

# import os
# import shutil
# from IPython.display import display

# SOURCE = "/kaggle/working/recovered_output/preprocessed_tutor_data.zip"
# DEST = "/kaggle/working/preprocessed_tutor_data.zip"

# # Copy the original preprocessed dataset
# if os.path.isfile(SOURCE):
#     shutil.copy2(SOURCE, DEST)
# else:
#     # Recreate ZIP from recovered dataset files
#     DATA_DIR = "/kaggle/working/recovered_output/data"
#     shutil.make_archive(
#         "/kaggle/working/preprocessed_tutor_data",
#         "zip",
#         DATA_DIR
#     )

# print("Dataset saved successfully!")
# print("Location:", DEST)
# print("Size (MB):", round(os.path.getsize(DEST) / 1024**2, 2))

# # Also create a ZIP of the recovered checkpoint
# CHECKPOINT_DIR = "/kaggle/working/recovered_output/checkpoints/checkpoint-1100"

# shutil.make_archive(
#     "/kaggle/working/checkpoint_1100_backup",
#     "zip",
#     CHECKPOINT_DIR
# )

# print("\nCheckpoint backup created:")
# print("/kaggle/working/checkpoint_1100_backup.zip")
# print("Size (MB):", round(
#     os.path.getsize("/kaggle/working/checkpoint_1100_backup.zip") / 1024**2, 2
# ))


Dataset saved successfully!
Location: /kaggle/working/preprocessed_tutor_data.zip
Size (MB): 9.69

Checkpoint backup created:
/kaggle/working/checkpoint_1100_backup.zip
Size (MB): 349.25


In [6]:

# # ============================================================
# # COMPLETE RECOVERY + EVALUATION + DOWNLOAD OUTPUTS
# # Qwen3-4B + LoRA checkpoint-1100 | Tesla T4
# # No retraining. Uses 4-bit quantization.
# # ============================================================

# import os, sys, json, math, gc, shutil, zipfile, subprocess
# from collections import Counter, defaultdict

# # 1. INSTALL / CLEANUP
# print("STEP 1: Preparing environment...", flush=True)

# subprocess.run(
#     [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
#     stdout=subprocess.DEVNULL,
#     stderr=subprocess.DEVNULL
# )
# subprocess.run(
#     [sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"],
#     check=True
# )

# import torch
# from transformers import (
#     AutoTokenizer,
#     AutoModelForCausalLM,
#     BitsAndBytesConfig
# )
# import peft
# from peft import PeftModel

# # Prevent PEFT from selecting the incompatible optional torchao backend.
# try:
#     import peft.import_utils as peft_import_utils
#     peft_import_utils.is_torchao_available = lambda: False

#     import peft.tuners.lora.torchao as peft_torchao
#     peft_torchao.is_torchao_available = lambda: False
# except Exception:
#     pass

# # 2. PATHS
# BASE_DIR = "/kaggle/working/recovered_output"
# CKPT_DIR = f"{BASE_DIR}/checkpoints/checkpoint-1100"
# TEST_FILE = f"{BASE_DIR}/data/test.jsonl"
# DATASET_ZIP = f"{BASE_DIR}/preprocessed_tutor_data.zip"

# assert torch.cuda.is_available(), "GPU is unavailable!"
# assert os.path.isdir(CKPT_DIR), "Checkpoint-1100 not found!"
# assert os.path.isfile(TEST_FILE), "Test dataset not found!"

# print("GPU:", torch.cuda.get_device_name(0))
# print("Checkpoint:", CKPT_DIR)

# # 3. DATASET
# print("\nSTEP 2: Loading test data...", flush=True)

# test_ex = []
# with open(TEST_FILE, encoding="utf-8") as f:
#     for line in f:
#         if line.strip():
#             test_ex.append(json.loads(line))

# print("Test examples:", len(test_ex))
# print("Source counts:", dict(Counter(x["source"] for x in test_ex)))

# # 4. TOKENIZER AND BASE MODEL
# print("\nSTEP 3: Loading tokenizer and 4-bit base model...", flush=True)

# with open(f"{CKPT_DIR}/adapter_config.json") as f:
#     adapter_config = json.load(f)

# BASE_MODEL = adapter_config["base_model_name_or_path"]
# print("Base model:", BASE_MODEL)

# tok = AutoTokenizer.from_pretrained(
#     CKPT_DIR,
#     trust_remote_code=True
# )

# if tok.pad_token is None:
#     tok.pad_token = tok.eos_token

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True
# )

# base_model = AutoModelForCausalLM.from_pretrained(
#     BASE_MODEL,
#     quantization_config=bnb_config,
#     device_map="auto",
#     low_cpu_mem_usage=True,
#     trust_remote_code=True
# )

# print("Base model loaded.")
# print("GPU memory (GB):", round(
#     torch.cuda.memory_allocated() / 1024**3, 2
# ))

# # 5. LOAD LORA ADAPTER
# print("\nSTEP 4: Loading LoRA checkpoint-1100...", flush=True)

# m = PeftModel.from_pretrained(
#     base_model,
#     CKPT_DIR,
#     is_trainable=False
# )
# m.eval()
# m.config.use_cache = True

# print("SUCCESS: LoRA adapter loaded!")

# # 6. GENERATION TEST
# print("\nSTEP 5: Testing generation...", flush=True)

# sources = [
#     "mathdial",
#     "socrateach",
#     "ncert_science",
#     "ncert_social_science",
#     "ncert_english",
#     "ncert_mathematics"
# ]

# generation_results = []

# for src in sources:
#     examples = [
#         x for x in test_ex
#         if x.get("source") == src
#         and isinstance(x.get("prompt"), list)
#         and len(x["prompt"]) > 0
#     ]

#     if not examples:
#         print("No examples:", src)
#         continue

#     e = examples[0]

#     try:
#         prompt_text = tok.apply_chat_template(
#             e["prompt"],
#             tokenize=False,
#             add_generation_prompt=True
#         )

#         inputs = tok(
#             prompt_text,
#             return_tensors="pt",
#             truncation=True,
#             max_length=1024
#         ).to("cuda:0")

#         with torch.inference_mode():
#             output = m.generate(
#                 **inputs,
#                 max_new_tokens=120,
#                 do_sample=False,
#                 pad_token_id=tok.pad_token_id
#             )

#         answer = tok.decode(
#             output[0][inputs["input_ids"].shape[1]:],
#             skip_special_tokens=True
#         )

#         reference = e.get("completion", "")
#         if isinstance(reference, list):
#             reference = "\n".join(
#                 str(v.get("content", v)) if isinstance(v, dict) else str(v)
#                 for v in reference
#             )

#         question = e["prompt"][-1].get("content", "")

#         print("\n" + "=" * 60)
#         print("SOURCE:", src)
#         print("QUESTION:", str(question)[:300])
#         print("REFERENCE:", str(reference)[:300])
#         print("MODEL ANSWER:", answer[:500])

#         generation_results.append({
#             "source": src,
#             "question": question,
#             "reference": str(reference),
#             "generated": answer
#         })

#         del inputs, output

#     except Exception as ex:
#         print("Generation error:", src, repr(ex))

#     gc.collect()
#     torch.cuda.empty_cache()

# # 7. LOSS AND PERPLEXITY
# print("\nSTEP 6: Calculating sampled test loss...", flush=True)

# MAX_PER_SOURCE = 20
# MAX_LENGTH = 1024
# loss_results = defaultdict(lambda: {
#     "weighted_loss": 0.0,
#     "tokens": 0,
#     "examples": 0
# })

# grouped = defaultdict(list)
# for e in test_ex:
#     grouped[e["source"]].append(e)

# for src, examples in grouped.items():
#     for e in examples[:MAX_PER_SOURCE]:
#         try:
#             prompt_ids = tok.apply_chat_template(
#                 e["prompt"],
#                 tokenize=True,
#                 add_generation_prompt=True
#             )

#             full_messages = e["prompt"] + e["completion"]
#             full_ids = tok.apply_chat_template(
#                 full_messages,
#                 tokenize=True,
#                 add_generation_prompt=False
#             )

#             if isinstance(prompt_ids, torch.Tensor):
#                 prompt_ids = prompt_ids.squeeze().tolist()
#             if isinstance(full_ids, torch.Tensor):
#                 full_ids = full_ids.squeeze().tolist()

#             # Only score completion tokens.
#             prompt_len = len(prompt_ids)
#             if full_ids[:prompt_len] != prompt_ids:
#                 # Skip if prompt tokenization does not align.
#                 continue

#             full_ids = full_ids[:MAX_LENGTH]
#             prompt_len = min(prompt_len, len(full_ids))

#             if len(full_ids) <= prompt_len:
#                 continue

#             input_ids = torch.tensor(
#                 [full_ids], dtype=torch.long, device="cuda:0"
#             )
#             labels = input_ids.clone()
#             labels[:, :prompt_len] = -100

#             with torch.inference_mode():
#                 outputs = m(
#                     input_ids=input_ids,
#                     attention_mask=torch.ones_like(input_ids),
#                     labels=labels
#                 )

#             valid_tokens = int((labels[:, 1:] != -100).sum().item())

#             if valid_tokens:
#                 loss_results[src]["weighted_loss"] += (
#                     float(outputs.loss.item()) * valid_tokens
#                 )
#                 loss_results[src]["tokens"] += valid_tokens
#                 loss_results[src]["examples"] += 1

#             del input_ids, labels, outputs

#         except Exception as ex:
#             print("Skipped loss example:", src, repr(ex))

#     gc.collect()
#     torch.cuda.empty_cache()

# # 8. METRICS
# print("\nSTEP 7: EVALUATION RESULTS")
# print("=" * 65)

# total_loss = 0.0
# total_tokens = 0
# source_metrics = {}

# for src, v in loss_results.items():
#     if not v["tokens"]:
#         continue

#     avg_loss = v["weighted_loss"] / v["tokens"]
#     ppl = math.exp(min(avg_loss, 700))

#     source_metrics[src] = {
#         "examples": v["examples"],
#         "tokens": v["tokens"],
#         "loss": avg_loss,
#         "perplexity": ppl
#     }

#     total_loss += v["weighted_loss"]
#     total_tokens += v["tokens"]

#     print(
#         f"{src:22s} | Examples: {v['examples']:2d} | "
#         f"Loss: {avg_loss:.4f} | Perplexity: {ppl:.4f}"
#     )

# overall_loss = total_loss / total_tokens if total_tokens else None
# overall_ppl = math.exp(min(overall_loss, 700)) if overall_loss is not None else None

# print("\nOVERALL LOSS:", overall_loss)
# print("OVERALL PERPLEXITY:", overall_ppl)
# print("Total scored examples:", sum(v["examples"] for v in loss_results.values()))

# # 9. SAVE EVALUATION REPORT
# print("\nSTEP 8: Saving reports and ZIP files...", flush=True)

# REPORT_PATH = "/kaggle/working/checkpoint_1100_evaluation.json"

# report = {
#     "base_model": BASE_MODEL,
#     "checkpoint": "checkpoint-1100",
#     "test_examples": len(test_ex),
#     "generation_results": generation_results,
#     "source_wise_metrics": source_metrics,
#     "overall_loss": overall_loss,
#     "overall_perplexity": overall_ppl
# }

# with open(REPORT_PATH, "w", encoding="utf-8") as f:
#     json.dump(report, f, indent=2, ensure_ascii=False)

# # Copy dataset ZIP into working directory.
# if os.path.isfile(DATASET_ZIP):
#     shutil.copy2(
#         DATASET_ZIP,
#         "/kaggle/working/preprocessed_tutor_data.zip"
#     )
# else:
#     shutil.make_archive(
#         "/kaggle/working/preprocessed_tutor_data",
#         "zip",
#         f"{BASE_DIR}/data"
#     )

# # Backup checkpoint folder.
# CHECKPOINT_ZIP = shutil.make_archive(
#     "/kaggle/working/checkpoint_1100_backup",
#     "zip",
#     CKPT_DIR
# )

# # Save generation examples separately.
# GEN_PATH = "/kaggle/working/generation_examples.json"
# with open(GEN_PATH, "w", encoding="utf-8") as f:
#     json.dump(generation_results, f, indent=2, ensure_ascii=False)

# print("\n" + "=" * 65)
# print("ALL COMPLETED")
# print("=" * 65)
# print("Dataset:", "/kaggle/working/preprocessed_tutor_data.zip")
# print("Checkpoint backup:", CHECKPOINT_ZIP)
# print("Evaluation report:", REPORT_PATH)
# print("Generation examples:", GEN_PATH)

# print("\nFILES READY IN KAGGLE OUTPUT:")
# for filename in [
#     "/kaggle/working/preprocessed_tutor_data.zip",
#     CHECKPOINT_ZIP,
#     REPORT_PATH,
#     GEN_PATH
# ]:
#     if os.path.isfile(filename):
#         print(os.path.basename(filename),
#               round(os.path.getsize(filename) / 1024**2, 2), "MB")


STEP 1: Preparing environment...
GPU: Tesla T4
Checkpoint: /kaggle/working/recovered_output/checkpoints/checkpoint-1100

STEP 2: Loading test data...
Test examples: 3324
Source counts: {'socrateach': 1569, 'mathdial': 1594, 'ncert_science': 97, 'ncert_english': 8, 'ncert_mathematics': 29, 'ncert_social_science': 27}

STEP 3: Loading tokenizer and 4-bit base model...
Base model: Qwen/Qwen3-4B-Instruct-2507


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Base model loaded.
GPU memory (GB): 1.9

STEP 4: Loading LoRA checkpoint-1100...
SUCCESS: LoRA adapter loaded!

STEP 5: Testing generation...

SOURCE: mathdial
QUESTION: The group produces an additional chair every 6 hours.
REFERENCE: So how many additional chairs are produced in the first 6 hours?
MODEL ANSWER: So how many additional chairs would be produced in 6 hours?

SOURCE: socrateach
QUESTION: The total cost of the field trip is $140, because each student pays $7, and there are 20 students.
REFERENCE: Well done! Now, if the field trip cost $140 and the class had $150 in total, how much is left in Miss Grayson's class fund after paying for the trip?
MODEL ANSWER: That's right! Now, if the class had $150 in total and the field trip cost $140, how much is left in Miss Grayson's class fund after the trip costs were paid?

SOURCE: ncert_science
QUESTION: Application Problem: Why are fibers like Nylon used to make ropes and carpets?
REFERENCE: Fibers have strong intermolecular forces 

In [7]:
# import os

# checkpoint_path = "/kaggle/working/recovered_output/checkpoints/checkpoint-1100"

# for file in sorted(os.listdir(checkpoint_path)):
#     print(file)

README.md
adapter_config.json
adapter_model.safetensors
chat_template.jinja
optimizer.pt
rng_state.pth
scaler.pt
scheduler.pt
tokenizer.json
tokenizer_config.json
trainer_state.json
training_args.bin
